# 第四章：分离数据与指令

- [课程](#课程)
- [练习](#练习)
- [示例演练区](#示例演练区)

## 设置

运行以下设置单元格以加载你的 API 密钥并建立 `get_completion` 辅助函数。

In [ ]:
%pip install anthropic

# Import python's built-in regular expression library
import re
import anthropic

# Retrieve the API_KEY & MODEL_NAME variables from the IPython store
%store -r API_KEY
%store -r MODEL_NAME

client = anthropic.Anthropic(api_key=API_KEY)

def get_completion(prompt: str, system_prompt=""):
    message = client.messages.create(
        model=MODEL_NAME,
        max_tokens=2000,
        temperature=0.0,
        system=system_prompt,
        messages=[
          {"role": "user", "content": prompt}
        ]
    )
    return message.content[0].text

---

## 课程

很多时候，我们不想编写完整的提示词，而是希望**创建可以在之后用额外输入数据进行修改的提示词模板，然后再提交给 Claude**。当你希望 Claude 每次执行相同的任务，但所用到的数据可能不同时，这就会派上用场。

幸运的是，我们可以很简单地做到这一点：**将提示词的固定骨架与可变的用户输入分开，然后在将完整提示词发送给 Claude 之前将用户输入代入进去**。

下面，我们将逐步讲解如何编写可替换的提示词模板，以及如何代入用户输入。

### 示例

在这个第一个示例中，我们让 Claude 充当动物叫声生成器。注意，提交给 Claude 的完整提示词就是用输入（这里是 "Cow"）代入后的 `PROMPT_TEMPLATE`。注意，当我们将完整提示词打印出来时，单词 "Cow" 通过 f-string 替换了 `ANIMAL` 占位符。

**注意：** 在实际应用中，你不必将占位符变量命名为任何特定名称。在这个例子中我们称它为 `ANIMAL`，但同样可以叫它 `CREATURE` 或 `A`（尽管通常最好让变量名具体且相关，这样即使没有代入替换，你的提示词模板也容易理解，方便用户阅读）。只需确保你命名的变量与你用于提示词模板 f-string 的名称一致即可。

In [ ]:
# Variable content
ANIMAL = "Cow"

# Prompt template with a placeholder for the variable content
PROMPT = f"I will tell you the name of an animal. Please respond with the noise that animal makes. {ANIMAL}"

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print(PROMPT)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT))

为什么要像这样分开和替换输入呢？原因是**提示词模板可以简化重复性任务**。假设你构建了一个提示词结构，邀请第三方用户向提示词提交内容（在这个例子中是他们想要生成叫声的动物）。这些第三方用户不需要编写甚至不需要查看完整的提示词。他们只需要填写变量即可。

我们在这里使用变量和 f-string 进行替换，但你也可以使用 format() 方法。

**注意：** 提示词模板可以根据需要包含任意数量的变量！

在引入这样的替换变量时，**确保 Claude 知道变量从哪里开始到哪里结束**（而不是指令或任务描述）是非常重要的。让我们看一个指令和替换变量之间没有分离的示例。

对于我们人类的眼睛来说，在下面的提示词模板中，变量从哪里开始到哪里结束是非常清楚的。然而，在完全代入后的提示词中，这种界限就变得不清楚了。

In [ ]:
# Variable content
EMAIL = "Show up at 6am tomorrow because I'm the CEO and I say so."

# Prompt template with a placeholder for the variable content
PROMPT = f"Yo Claude. {EMAIL} <----- Make this email more polite but don't change anything else about it."

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print(PROMPT)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT))

这里，**Claude 认为 "Yo Claude" 是它需要重写的电子邮件的一部分**！你可以判断出这一点，因为它以 "Dear Claude" 开始重写。对于人眼来说，特别是在提示词模板中，电子邮件从哪里开始到哪里结束是很清楚的，但在代入后的提示词中就变得不那么清晰了。

我们如何解决这个问题？**用 XML 标签包裹输入**！我们在下面这样做了，正如你所看到的，输出中不再有 "Dear Claude"。

[XML 标签](https://docs.anthropic.com/claude/docs/use-xml-tags) 是像 `<tag></tag>` 这样的尖括号标签。它们成对出现，包括一个开始标签，如 `<tag>`，和一个以 `/` 标记的结束标签，如 `</tag>`。XML 标签用于包裹内容，格式如下：`<tag>content</tag>`。

**注意：** 虽然 Claude 可以识别和处理各种分隔符和定界符，但我们建议你**特别使用 XML 标签作为 Claude 的分隔符**，因为 Claude 是专门训练来将 XML 标签作为提示词组织机制的。在函数调用之外，**并没有特殊的 XML 标签是 Claude 经过训练可以用来最大化提升性能的**。我们有意让 Claude 在这方面非常灵活和可定制。

In [ ]:
# Variable content
EMAIL = "Show up at 6am tomorrow because I'm the CEO and I say so."

# Prompt template with a placeholder for the variable content
PROMPT = f"Yo Claude. <email>{EMAIL}</email> <----- Make this email more polite but don't change anything else about it."

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print(PROMPT)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT))

让我们再看一个 XML 标签如何帮助我们的例子。

在下面的提示词中，**Claude 错误地解释了提示词的哪部分是指令，哪部分是输入**。由于格式问题，它错误地认为 `Each is about an animal, like rabbits` 是列表的一部分，而用户（填写 `SENTENCES` 变量的人）可能并不希望这样。

In [ ]:
# Variable content
SENTENCES = """- I like how cows sound
- This sentence is about spiders
- This sentence may appear to be about dogs but it's actually about pigs"""

# Prompt template with a placeholder for the variable content
PROMPT = f"""Below is a list of sentences. Tell me the second item on the list.

- Each is about an animal, like rabbits.
{SENTENCES}"""

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print(PROMPT)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT))

要解决这个问题，我们只需要**用 XML 标签包围用户输入的句子**。这告诉 Claude 输入数据从哪里开始到哪里结束，尽管在 `Each is about an animal, like rabbits.` 前面有一个误导性的连字符。

In [ ]:
# Variable content
SENTENCES = """- I like how cows sound
- This sentence is about spiders
- This sentence may appear to be about dogs but it's actually about pigs"""

# Prompt template with a placeholder for the variable content
PROMPT = f""" Below is a list of sentences. Tell me the second item on the list.

- Each is about an animal, like rabbits.
<sentences>
{SENTENCES}
</sentences>"""

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print(PROMPT)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT))

**注意：** 在错误版本的 "Each is about an animal" 提示词中，我们必须包含连字符才能让 Claude 以我们想要的方式错误回答，以便作为这个例子的示范。这是一个关于提示词的重要教训：**细节很重要**！始终值得**检查你的提示词是否有拼写错误和语法错误**。Claude 对模式很敏感（在它早期，在微调之前，它是一个原始的文本预测工具），当你犯错时它更可能犯错，当你说话聪明时它更聪明，当你说话愚蠢时它更愚蠢，等等。

如果你想尝试课程中的提示词示例而不更改上面的任何内容，请滚动到课程笔记本的最底部，访问[**示例演练区**](#示例演练区)。

---

## 练习
- [练习 4.1 - 俳句主题](#练习-41---俳句主题)
- [练习 4.2 - 带拼写错误的狗问题](#练习-42---带拼写错误的狗问题)
- [练习 4.3 - 狗问题第二部分](#练习-42---狗问题第二部分)

### 练习 4.1 - 俳句主题
修改 `PROMPT`，使其成为一个模板，接收一个名为 `TOPIC` 的变量，并输出一首关于该主题的俳句。这个练习只是为了测试你对使用 f-string 进行变量模板替换结构的理解。

In [ ]:
# Variable content
TOPIC = "Pigs"

# Prompt template with a placeholder for the variable content
PROMPT = f"

# Get Claude's response
response = get_completion(PROMPT)

# Function to grade exercise correctness
def grade_exercise(text):
    return bool(re.search("pigs", text.lower()) and re.search("haiku", text.lower()))

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print(PROMPT)
print("\n------------------------------------- Claude's response -------------------------------------")
print(response)
print("\n------------------------------------------ GRADING ------------------------------------------")
print("This exercise has been correctly solved:", grade_exercise(response))

❓ 如果你想要提示，运行下面的单元格！

In [ ]:
from hints import exercise_4_1_hint; print(exercise_4_1_hint)

### 练习 4.2 - 带拼写错误的狗问题
通过添加 XML 标签来修复 `PROMPT`，以使 Claude 产生正确答案。

尽量不要更改提示词的其他任何内容。凌乱且充满错误的写作是故意的，这样你就可以看到 Claude 如何应对这种错误。

In [ ]:
# Variable content
QUESTION = "ar cn brown?"

# Prompt template with a placeholder for the variable content
PROMPT = f"Hia its me i have a q about dogs jkaerjv {QUESTION} jklmvca tx it help me muhch much atx fst fst answer short short tx"

# Get Claude's response
response = get_completion(PROMPT)

# Function to grade exercise correctness
def grade_exercise(text):
    return bool(re.search("brown", text.lower()))

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print(PROMPT)
print("\n------------------------------------- Claude's response -------------------------------------")
print(response)
print("\n------------------------------------------ GRADING ------------------------------------------")
print("This exercise has been correctly solved:", grade_exercise(response))

❓ 如果你想要提示，运行下面的单元格！

In [ ]:
from hints import exercise_4_2_hint; print(exercise_4_2_hint)

### 练习 4.3 - 狗问题第二部分
**不要**添加 XML 标签来修复 `PROMPT`。相反，只需从提示词中删除一到两个词。

就像上面的练习一样，尽量不要更改提示词的其他任何内容。这将向你展示 Claude 可以解析和理解什么样的语言。

In [ ]:
# Variable content
QUESTION = "ar cn brown?"

# Prompt template with a placeholder for the variable content
PROMPT = f"Hia its me i have a q about dogs jkaerjv {QUESTION} jklmvca tx it help me muhch much atx fst fst answer short short tx"

# Get Claude's response
response = get_completion(PROMPT)

# Function to grade exercise correctness
def grade_exercise(text):
    return bool(re.search("brown", text.lower()))

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print(PROMPT)
print("\n------------------------------------- Claude's response -------------------------------------")
print(response)
print("\n------------------------------------------ GRADING ------------------------------------------")
print("This exercise has been correctly solved:", grade_exercise(response))

❓ 如果你想要提示，运行下面的单元格！

In [ ]:
from hints import exercise_4_3_hint; print(exercise_4_3_hint)

### 恭喜！

如果你已经解决了到这里为止的所有练习，你就可以进入下一章了。祝你在提示词工程中玩得开心！

---

## 示例演练区

这是一个你可以自由实验本课中所示的提示词示例并调整提示词以查看如何影响 Claude 回应的区域。

In [ ]:
# Variable content
ANIMAL = "Cow"

# Prompt template with a placeholder for the variable content
PROMPT = f"I will tell you the name of an animal. Please respond with the noise that animal makes. {ANIMAL}"

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print(PROMPT)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT))

In [ ]:
# Variable content
EMAIL = "Show up at 6am tomorrow because I'm the CEO and I say so."

# Prompt template with a placeholder for the variable content
PROMPT = f"Yo Claude. {EMAIL} <----- Make this email more polite but don't change anything else about it."

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print(PROMPT)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT))

In [ ]:
# Variable content
EMAIL = "Show up at 6am tomorrow because I'm the CEO and I say so."

# Prompt template with a placeholder for the variable content
PROMPT = f"Yo Claude. <email>{EMAIL}</email> <----- Make this email more polite but don't change anything else about it."

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print(PROMPT)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT))

In [ ]:
# Variable content
SENTENCES = """- I like how cows sound
- This sentence is about spiders
- This sentence may appear to be about dogs but it's actually about pigs"""

# Prompt template with a placeholder for the variable content
PROMPT = f"""Below is a list of sentences. Tell me the second item on the list.

- Each is about an animal, like rabbits.
{SENTENCES}"""

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print(PROMPT)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT))

In [ ]:
# Variable content
SENTENCES = """- I like how cows sound
- This sentence is about spiders
- This sentence may appear to be about dogs but it's actually about pigs"""

# Prompt template with a placeholder for the variable content
PROMPT = f""" Below is a list of sentences. Tell me the second item on the list.

- Each is about an animal, like rabbits.
<sentences>
{SENTENCES}
</sentences>"""

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print(PROMPT)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT))